In [ ]:
import  osiris_utils as ou
from matplotlib import pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FuncFormatter
import matplotlib.colors as colors
from pathlib import Path

plt.rcParams['font.size'] = 14


In [ ]:
def createSimDic(path, sim_labels, test):
    sim = {}
    for key in sim_labels.keys():
        sim[key] = {}
        for dtw in sim_labels[key]:
            sim[key][dtw] = ou.Simulation(f"{path}/{test}/{key}/dtw{dtw}/{key}.in")
    return sim


def createSimDic_nx(path, sim_labels, test):
    sim = {}
    for key in sim_labels.keys():
        sim[key] = {}
        for dtw in sim_labels[key]:
            sim[key][dtw] = ou.Simulation(f"{path}/{test}/{key}/nx{dtw}/{key}.in")
    return sim

def createRawDic_nx(path, sim_labels, test):
    rawdic = {}
    for key in sim_labels.keys():
        rawdic[key] = {}
        for dtw in sim_labels[key]:
            raw_dir = Path(path) / test / key / f"nx{dtw}" / "MS" / "RAW" / "test_electrons"
            raw_files = sorted(raw_dir.glob("RAW-test_electrons-*.h5"))
            if not raw_files:
                raise FileNotFoundError(f"No RAW files found in {raw_dir}")
            rawdic[key][dtw] = [ou.OsirisRawFile(raw_file) for raw_file in raw_files]
    return rawdic

In [ ]:
# Normalize axis to w_ce


def _set_scaled_formatter(axis, scale_factor, fmt=".2f"):
    axis.set_major_formatter(
        FuncFormatter(lambda v, pos: f"{v*scale_factor:{fmt}}")
    )

def scale_x_ax(scale_factor, fig, ax, label=r"$t[1 / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_xlabel(label)
    if lock_ticks:
        # freeze current tick positions
        ticks = ax.get_xticks()
        ax.xaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    return fig, ax

def scale_y_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_ylabel(label)
    if lock_ticks:
        ticks = ax.get_yticks()
        ax.yaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    return fig, ax

def scale_z_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_zlabel(label)
    if lock_ticks:
        ticks = ax.get_zticks()
        ax.zaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax

def scale_3d_axes(scale_factor, fig, ax, fmt=".2f"):
    # ax.set_xlabel(r"$x_1[c / \Omega_e]$")
    # ax.set_ylabel(r"$x_2[c / \Omega_e]$")
    # ax.set_zlabel(r"$x_3[c / \Omega_e]$")
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax


In [ ]:
def sort_raw_by_tag(raw):
    tag = raw.data["tag"]
    order = np.lexsort((tag[:, 1], tag[:, 0]))

    for key, arr in raw.data.items():
        if hasattr(arr, "shape") and arr.shape[0] == len(order):
            raw.data[key] = arr[order]

    return order


In [ ]:
class curvDriftTheo:
    def __init__(self, sim, B=None, rqm = -1, direc = -1):
        self.rqm = rqm
        self.direc = direc
        self.sim = sim
        self.x1_0 = sim["test_electrons"]["tracks"]["x1"][:,0]
        self.x2_0 = sim["test_electrons"]["tracks"]["x2"][:,0]
        self.x3_0 = sim["test_electrons"]["tracks"]["x3"][:,0]
        self.phi0 = np.arctan2(self.x2_0, self.x1_0)
        self.p1_0 = sim["test_electrons"]["tracks"]["p1"][:,0]
        self.p2_0 = sim["test_electrons"]["tracks"]["p2"][:,0]
        self.p3_0 = sim["test_electrons"]["tracks"]["p3"][:,0]
        self.gamma_0 = np.sqrt(1 + self.p1_0**2 + self.p2_0**2 + self.p3_0**2)
        
        self.R0 = np.sqrt(sim["test_electrons"]["tracks"]["x1"][:,0]**2 + sim["test_electrons"]["tracks"]["x2"][:,0]**2)

        if B is not None:
            self.B0 = B
        else:
            self.B0 = np.sqrt(sim["test_electrons"]["tracks"]["B1"][:,0]**2 + sim["test_electrons"]["tracks"]["B2"][:,0]**2 + sim["test_electrons"]["tracks"]["B3"][:,0]**2)

        self.vc, self.v_par = self._curv_v()

    def b1(self, x1, x2, x3):
        return self.B0 * (-x2) / np.sqrt(x1**2 + x2**2)
    def b2(self, x1, x2, x3):
        return self.B0 * x1 / np.sqrt(x1**2 + x2**2)
    def b3(self, x1, x2, x3):
        return np.zeros_like(x1)

    def _curv_v(self):
        sim = self.sim


        p_par0 = (self.p1_0 * self.b1(self.x1_0, self.x2_0, self.x3_0) + \
                self.p2_0 * self.b2(self.x1_0, self.x2_0, self.x3_0) + \
                self.p3_0 * self.b3(self.x1_0, self.x2_0, self.x3_0) ) / self.B0
        
        v_par = p_par0 / self.gamma_0
        vc = self.rqm * p_par0**2 / self.B0 / self.gamma_0 * self.direc / self.R0

        return vc, v_par

    def get_curv_traj(self, t):
        t = np.asarray(t, dtype=float)              # shape: (nt,)
        omega = self.v_par / self.R0      # shape: (npart,)

        x3 = self.x3_0[:, None] + np.outer(self.vc, t)

        phase = np.outer(omega, t) + self.phi0[:, None]

        x1 = self.R0[:, None] * np.cos(phase)
        x2 = self.R0[:, None] * np.sin(phase)

        return np.array([x1, x2, x3])
    



class curvDriftTheo_Raw:
    def __init__(self, raws, B, rqm = -1, direc = -1):
        self.rqm = rqm
        self.direc = direc
        self.raws = raws
        self.x1_0 = raws[0].data["x1"]
        self.x2_0 = raws[0].data["x2"]
        self.x3_0 = raws[0].data["x3"]
        self.phi0 = np.arctan2(self.x2_0, self.x1_0)
        self.p1_0 = raws[0].data["p1"]
        self.p2_0 = raws[0].data["p2"]
        self.p3_0 = raws[0].data["p3"]
        self.gamma_0 = np.sqrt(1 + self.p1_0**2 + self.p2_0**2 + self.p3_0**2)
        
        self.R0 = np.sqrt(self.x1_0**2 + self.x2_0**2)

        self.B0 = B

        self.vc, self.v_par = self._curv_v()

    def b1(self, x1, x2, x3):
        return self.B0 * (-x2) / np.sqrt(x1**2 + x2**2)
    def b2(self, x1, x2, x3):
        return self.B0 * x1 / np.sqrt(x1**2 + x2**2)
    def b3(self, x1, x2, x3):
        return np.zeros_like(x1)

    def _curv_v(self):

        p_par0 = (self.p1_0 * self.b1(self.x1_0, self.x2_0, self.x3_0) + \
                self.p2_0 * self.b2(self.x1_0, self.x2_0, self.x3_0) + \
                self.p3_0 * self.b3(self.x1_0, self.x2_0, self.x3_0) ) / self.B0
        
        v_par = p_par0 / self.gamma_0
        vc = self.rqm * p_par0**2 / self.B0 / self.gamma_0 * self.direc / self.R0

        return vc, v_par

    def get_curv_traj(self, t):
        t = float(t)
        omega = self.v_par / self.R0      # shape: (npart,)

        x3 = self.x3_0 + self.vc * t

        phase = omega * t + self.phi0

        x1 = self.R0 * np.cos(phase)
        x2 = self.R0 * np.sin(phase)

        return np.array([x1, x2, x3])
    

In [ ]:
sim = ou.Simulation("/home/exxxx5/Tese/Decks/weibelTestsFinalV2/E0_01HOT_FinalV/Boris/dtw0_01/Boris.in")

In [ ]:
test = "Curv"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}

B = 1000.
sim = createSimDic(path, sim_labels, test)

In [ ]:
t = sim["Gca"]["1000"]["test_electrons"]["tracks"]["t"][0,1]
traj_theo = curvDriftTheo(sim["Gca"]["1000"], B, direc=1).get_curv_traj(t)[:,:,0]
radius_theo = curvDriftTheo(sim["Gca"]["1000"], B, direc=1).R0
print("theo radius:", radius_theo)
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

from matplotlib.lines import Line2D

fig, ax = plt.subplots()
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        radius = np.sqrt(x1**2 + x2**2)

        err = np.abs(radius - radius_theo)/ radius_theo

        mean = np.mean(err)
        std = np.std(err, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    eb = ax.errorbar(
        X, Y, yerr=YERR,
        label=fr"{pusher}",
        fmt='o',
        markersize=5,
        linestyle=linestyle,
        linewidth=1.2,
        capsize=3,
        elinewidth=1.0,
    )

    ax.plot(
        X,
        YMax,
        marker='x',
        linestyle='none',
        markersize=5,
        markeredgewidth=1.0,
        zorder=3,
        color=eb.lines[0].get_color(),
    )

ax.set_ylabel("relative radial error")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
ax.set_title("t = " + str(round(t * B / 2.0 / np.pi, 1)) + r" $[2\pi / \Omega_e]$")
ax.legend()
ax.set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
    # 'GcaHighRes': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    # 'gcaNoDrifts': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'gcaNoDrifts_doublePrecDiag_and_Gca': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
    'gcaNoDrifts_doublePrecDiag_and_Gca_maxIter1000': ["1000", "500", "100", "50", "10", "1", "0_1", "0_01", "0_001", "0_0001"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

machine_pre = 10**-16

components = ["r", "x3"]
pusher_labels = {
    "Gca": "GCA",
    "GcaHighRes": "GCA high res",
    "gcaNoDrifts": "GCA no drifts",
    "gcaNoDrifts_doublePrecDiag_and_Gca": "GCA high res no drifts",
    "gcaNoDrifts_doublePrecDiag_and_Gca_maxIter1000": "GCA high res no drifts maxIter1000",
}

fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
mach_prec_plotted = [False] * len(axes)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    mach_prec = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))
        mach_prec.append(machine_pre / L / num_steps)

    if pusher == "GcaHighRes":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)
    MachPrec = np.asarray(mach_prec, dtype=float)
    for i, ax in enumerate(axes):
        if not mach_prec_plotted[i]:
            ax.plot(
                X,
                MachPrec,
                marker='x',
                linestyle=':',
                markersize=0,
                markeredgewidth=1.0,
                zorder=3,
                label="double machine precision",
                color="gray",
            )
            mach_prec_plotted[i] = True
                
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher_labels.get(pusher, pusher)}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        # ax.set_ylabel(fr"{components[i]} error dt L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_DoublePrec"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaHighRes"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaOnlyCurvDrift"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoGradB"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoGradBInit"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBInit"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradBInit = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradB"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradB = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBdrift"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradBdrift = createSimDic(path, sim_labels, test)

In [ ]:
test_sims = {
    "All drifts": sim_Curv_1step,
    # "No grad B init": sim_Curv_1step_GcaNoGradBInit,
    # "No grad B drift": sim_Curv_1step_GcaNoGradBdrift,
    "No grad B": sim_Curv_1step_GcaNoGradB,
}

pusher = "Gca"
grid = test_sims["All drifts"][pusher]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for test_label, sim in test_sims.items():
    X = []
    Y = []
    YERR = []
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        num_steps = float(dtw.replace("_", ".")) / 1000
        err = (np.abs(traj - traj_theo)) / L / num_steps

        err_radial = np.sqrt(err[0]**2 + err[1]**2)
        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)
    order = np.argsort(X)
    X = X[order]
    Y = Y[order]
    YERR = YERR[order]
    YMax = YMax[order]

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=test_label,
            fmt='o',
            markersize=5,
            linestyle='-',
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend(title="Gca pusher")
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GradBDiag"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
sim["Gca"]["1000"]["test_electrons"]["tracks"]["dB1dx1"]

def get_dBdx(sim):
    return np.array([
        [
            sim["test_electrons"]["tracks"]["dB1dx1"],
            sim["test_electrons"]["tracks"]["dB1dx2"],
            sim["test_electrons"]["tracks"]["dB1dx3"],
        ],
        [
            sim["test_electrons"]["tracks"]["dB2dx1"],
            sim["test_electrons"]["tracks"]["dB2dx2"],
            sim["test_electrons"]["tracks"]["dB2dx3"],
        ],
        [
            sim["test_electrons"]["tracks"]["dB3dx1"],
            sim["test_electrons"]["tracks"]["dB3dx2"],
            sim["test_electrons"]["tracks"]["dB3dx3"],
        ],
    ])

def get_B(sim):
    return np.array([
        sim["test_electrons"]["tracks"]["B1"],
        sim["test_electrons"]["tracks"]["B2"],
        sim["test_electrons"]["tracks"]["B3"],
    ])

def get_traj(sim):
    return np.array([
        sim["test_electrons"]["tracks"]["x1"],
        sim["test_electrons"]["tracks"]["x2"],
        sim["test_electrons"]["tracks"]["x3"],
    ])

B_theo = 1000
dbdx = get_dBdx(sim["Gca"]["1000"])
traj = get_traj(sim["Gca"]["1000"])
B_vec = get_B(sim["Gca"]["1000"])
B_mag = np.linalg.norm(B_vec, axis=0)

gradB = np.einsum("jnt,jint->int", B_vec, dbdx) / B_mag
gradB_mag = np.linalg.norm(gradB, axis=0)

In [ ]:
gradB_mag_normalized = gradB_mag / B_theo / B_theo

plt.figure(figsize=(8, 6))
sc = plt.scatter(
    traj[0].flatten(),
    traj[1].flatten(),
    c=gradB_mag_normalized.flatten(),
    s=2,
    cmap="viridis"
)
plt.colorbar(sc, label=r"$|\nabla B| / B^2 \, [e / c^2 m_e]$")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

plt.figure(figsize=(8, 6))
sc = plt.scatter(
    traj[0].flatten(),
    traj[1].flatten(),
    c=np.abs(B_mag.flatten()-B_theo)/B_theo,
    s=2,
    cmap="viridis"
)
plt.colorbar(sc, label=r"Relative error in $|B|$")
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

In [ ]:
test = "Curv_1step_GcaInitCurvOnly"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaOnlyCurvDriftV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaGradBOnly"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoGradBInitV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoGradBV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1stepV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1stepV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBInitV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradBInit = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradB = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaNoGradBdriftV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaNoGradBdrift = createSimDic(path, sim_labels, test)




test = "Curv_1step_GcaOnlyCurvDriftV2"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Gca': ["1000", "100", "10", "1", "0_1"],   
}
sim_Curv_1step_GcaOnlyCurvDrift = createSimDic(path, sim_labels, test)

In [ ]:
test_sims = {
    "All drifts": sim_Curv_1step,
    "No grad B init": sim_Curv_1step_GcaNoGradBInit,
    "No grad B drift": sim_Curv_1step_GcaNoGradBdrift,
    "Neither": sim_Curv_1step_GcaNoGradB,
    "Only curv drift": sim_Curv_1step_GcaOnlyCurvDrift,
}

pusher = "Gca"
grid = test_sims["All drifts"][pusher]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for test_label, sim in test_sims.items():
    X = []
    Y = []
    YERR = []
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        num_steps = float(dtw.replace("_", ".")) / 1000
        err = (np.abs(traj - traj_theo)) / L / num_steps

        err_radial = np.sqrt(err[0]**2 + err[1]**2)
        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)
    order = np.argsort(X)
    X = X[order]
    Y = Y[order]
    YERR = YERR[order]
    YMax = YMax[order]

    for i, ax in enumerate(axes):
        if test_label == "All drifts" or test_label == "No grad B init":
            linestyle = '-'
        elif test_label == "Only curv drift":
            linestyle = ':'
        else:            linestyle = '--'

        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=test_label,
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend(title="Test")
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoDrifts"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    # 'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "100", "10", "1", "0_1"],   
    # 'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000, direc=1).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        # num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error dt / dt_max / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoDrifts_dxStudy"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'gcaNoDrifts': ["24", "40", "60", "80", "100", "120", "140", "160", "180"],
    # 'gcaCurvOnly': ["24", "40", "60", "80", "100", "120", "140", "160", "180"],
    'gcaNoDrifts1ppc': ["24", "40", "60", "80", "100", "120", "140", "160", "180"],
}


sim = createSimDic_nx(path, sim_labels, test)

rawdic = createRawDic_nx(path, sim_labels, test)

In [ ]:
tag0 = rawdic["gcaNoDrifts"]["80"][0].data["tag"]
tag1 = rawdic["gcaNoDrifts"]["80"][1].data["tag"]

print(np.array_equal(tag0, tag1))

tag0 = rawdic["gcaNoDrifts1ppc"]["80"][0].data["tag"]
tag1 = rawdic["gcaNoDrifts1ppc"]["80"][1].data["tag"]

print(np.array_equal(tag0, tag1))

In [ ]:
raw_8ppc = rawdic["gcaNoDrifts"]["80"]
raw_1ppc = rawdic["gcaNoDrifts1ppc"]["80"]
raw_index_8ppc = 1
raw_index_1ppc = 1

grid = raw_8ppc[0].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]
num_steps = 1


def debug_ranges(label, **arrays):
    print(f"\n{label}")
    for name, values in arrays.items():
        values = np.asarray(values)
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            print(f"  {name}: shape={values.shape}, no finite values")
            continue
        print(
            f"  {name}: shape={values.shape}, "
            f"min={finite.min():.6g}, max={finite.max():.6g}, "
            f"mean={finite.mean():.6g}"
        )


# ---------- 8 ppc ----------
theo_8ppc = curvDriftTheo_Raw(raw_8ppc, 1000, direc=1)
t_8ppc_1 = raw_8ppc[raw_index_8ppc].time[0]


print(f"Time for 8 ppc: {t_8ppc_1}")
traj_theo_8ppc_1 = theo_8ppc.get_curv_traj(t_8ppc_1)

x1_8ppc_1 = raw_8ppc[raw_index_8ppc].data["x1"]
x2_8ppc_1 = raw_8ppc[raw_index_8ppc].data["x2"]
x3_8ppc_1 = raw_8ppc[raw_index_8ppc].data["x3"]
traj_8ppc_1 = np.array([x1_8ppc_1, x2_8ppc_1, x3_8ppc_1])

err_8ppc = np.abs(traj_8ppc_1 - traj_theo_8ppc_1) / L / num_steps
err_radial_8ppc = np.sqrt(err_8ppc[0]**2 + err_8ppc[1]**2)

x1_8ppc_0 = raw_8ppc[0].data["x1"]
x2_8ppc_0 = raw_8ppc[0].data["x2"]
r0_8ppc = np.sqrt(x1_8ppc_0**2 + x2_8ppc_0**2)

p1_8ppc_0 = raw_8ppc[0].data["p1"]
p2_8ppc_0 = raw_8ppc[0].data["p2"]

b1_theo_8ppc_0 = theo_8ppc.b1(x1_8ppc_0, x2_8ppc_0, 0)
b2_theo_8ppc_0 = theo_8ppc.b2(x1_8ppc_0, x2_8ppc_0, 0)

p_par_8ppc = (p1_8ppc_0*b1_theo_8ppc_0 + p2_8ppc_0*b2_theo_8ppc_0) / 1000


# ---------- 1 ppc ----------
theo_1ppc = curvDriftTheo_Raw(raw_1ppc, 1000, direc=1)
t_1ppc_1 = raw_1ppc[raw_index_1ppc].time[0]
traj_theo_1ppc_1 = theo_1ppc.get_curv_traj(t_1ppc_1)

x1_1ppc_1 = raw_1ppc[raw_index_1ppc].data["x1"]
x2_1ppc_1 = raw_1ppc[raw_index_1ppc].data["x2"]
x3_1ppc_1 = raw_1ppc[raw_index_1ppc].data["x3"]
traj_1ppc_1 = np.array([x1_1ppc_1, x2_1ppc_1, x3_1ppc_1])

err_1ppc = np.abs(traj_1ppc_1 - traj_theo_1ppc_1) / L / num_steps
err_radial_1ppc = np.sqrt(err_1ppc[0]**2 + err_1ppc[1]**2)

x1_1ppc_0 = raw_1ppc[0].data["x1"]
x2_1ppc_0 = raw_1ppc[0].data["x2"]
r0_1ppc = np.sqrt(x1_1ppc_0**2 + x2_1ppc_0**2)

p1_1ppc_0 = raw_1ppc[0].data["p1"]
p2_1ppc_0 = raw_1ppc[0].data["p2"]

b1_theo_1ppc_0 = theo_1ppc.b1(x1_1ppc_0, x2_1ppc_0, 0)
b2_theo_1ppc_0 = theo_1ppc.b2(x1_1ppc_0, x2_1ppc_0, 0)

p_par_1ppc = (p1_1ppc_0*b1_theo_1ppc_0 + p2_1ppc_0*b2_theo_1ppc_0) / 1000


print(f"L = {L:.6g}, num_steps = {num_steps}")
print(f"particle counts: 8ppc={len(r0_8ppc)}, 1ppc={len(r0_1ppc)}")
debug_ranges(
    "8ppc initial ranges",
    r0=r0_8ppc,
    p1=p1_8ppc_0,
    p2=p2_8ppc_0,
    b1_theo=b1_theo_8ppc_0,
    b2_theo=b2_theo_8ppc_0,
    p_par=p_par_8ppc,
    err_radial=err_radial_8ppc,
)
debug_ranges(
    "1ppc initial ranges",
    r0=r0_1ppc,
    p1=p1_1ppc_0,
    p2=p2_1ppc_0,
    b1_theo=b1_theo_1ppc_0,
    b2_theo=b2_theo_1ppc_0,
    p_par=p_par_1ppc,
    err_radial=err_radial_1ppc,
)


error_floor = 1e-15

positive_err = np.concatenate([
    err_radial_8ppc[err_radial_8ppc >= 0],
    err_radial_1ppc[err_radial_1ppc >= 0],
])

if positive_err.size == 0:
    raise ValueError("Log color scale needs at least one positive radial error")

vmin = error_floor
vmax = positive_err.max()

norm = colors.LogNorm(vmin=vmin, vmax=vmax)
err_radial_8ppc_for_color = np.maximum(err_radial_8ppc, error_floor)
err_radial_1ppc_for_color = np.maximum(err_radial_1ppc, error_floor)

fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    r0_1ppc,
    p_par_1ppc,
    label="1ppc",
    alpha=0.5,
    marker="x",
    c=err_radial_1ppc_for_color,
    cmap="viridis",
    norm=norm,
)

ax.set_xlabel("Initial radius")
ax.set_ylabel(r"$p_{\parallel 0}$")
ax.legend()
plt.colorbar(sc, ax=ax, label="Radial error / L")
plt.show()


fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    r0_8ppc,
    p_par_8ppc,
    label="8ppc",
    alpha=0.5,
    c=err_radial_8ppc_for_color,
    cmap="viridis",
    norm=norm,
)

ax.set_xlabel("Initial radius")
ax.set_ylabel(r"$p_{\parallel 0}$")
ax.legend()
plt.colorbar(sc, ax=ax, label="Radial error / L")
plt.show()




fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(r0_8ppc, np.maximum(err_radial_8ppc, error_floor), label="8ppc", alpha=0.5)
ax.scatter(r0_1ppc, np.maximum(err_radial_1ppc, error_floor), label="1ppc", alpha=0.5, marker='x')
ax.axhline(error_floor, color="gray", linestyle=":", linewidth=1, label=r"plot floor")
ax.set_xlabel("Initial radius")
ax.set_ylabel("Radial error")
ax.legend()
ax.set_yscale("log")
plt.show()

# ---------- log data + same bins ----------
err_radial_8ppc_nonzero = err_radial_8ppc[err_radial_8ppc > 0]
err_radial_1ppc_nonzero = err_radial_1ppc[err_radial_1ppc > 0]

log_err_8ppc = np.log10(err_radial_8ppc_nonzero)
log_err_1ppc = np.log10(err_radial_1ppc_nonzero)

log_min = min(log_err_8ppc.min(), log_err_1ppc.min())
log_max = max(log_err_8ppc.max(), log_err_1ppc.max())

bins = np.linspace(log_min, log_max, 100)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(log_err_8ppc, bins=bins, alpha=0.5, label="8ppc", log=True)
ax.hist(log_err_1ppc, bins=bins, alpha=0.5, label="1ppc", log=True)

ax.set_xlabel(r"$\log_{10}(\mathrm{radial\ error})$")
ax.set_ylabel("Count")
ax.legend()

plt.show()


In [ ]:
mask = (err_radial_1ppc < 10**(-13))
particles_low_err = np.where(mask)[0]

raw_1ppc = rawdic["gcaNoDrifts1ppc"]["80"]
raw0_1ppc = raw_1ppc[0]
raw_data_1ppc = raw0_1ppc.data

x1_base = raw_data_1ppc["x1"]
x2_base = raw_data_1ppc["x2"]
p1_base = raw_data_1ppc["p1"]
p2_base = raw_data_1ppc["p2"]

x1 = x1_base[particles_low_err]
x2 = x2_base[particles_low_err]
p1 = p1_base[particles_low_err]
p2 = p2_base[particles_low_err]

theo = curvDriftTheo_Raw(raw_1ppc, 1000, direc=1)
p_par = (p1*theo.b1(x1, x2, 0) + p2*theo.b2(x1, x2, 0)) / 1000
p_par_base = (p1_base*theo.b1(x1_base, x2_base, 0) + p2_base*theo.b2(x1_base, x2_base, 0)) / 1000

# p_par_sim = (p1*b1 + p2*b2) / np.sqrt(b1**2 + b2**2)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x1_base, x2_base, alpha=0.5, label="All particles 1ppc")
ax.scatter(x1, x2, alpha=0.5, label="Low error")

xmin = min(x1_base.min(), x2_base.min())
xmax = max(x1_base.max(), x2_base.max())
xx = np.linspace(xmin, xmax, 200)
ax.plot(xx, -xx, color="red", linestyle="--", label=r"$x_2=-x_1$")

ax.set_xlabel("x1_0")
ax.set_ylabel("x2_0")
ax.set_title("Particles with low radial error 1ppc")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(p1_base, p2_base, alpha=0.5, label="All particles 1ppc")
ax.scatter(p1, p2, alpha=0.5, label="Low error")

ax.set_xlabel("p1_0")
ax.set_ylabel("p2_0")
ax.set_title("Particles with low radial error 1ppc")
ax.legend()
plt.show()



p_par = np.asarray(p_par).ravel()
p_par_base = np.asarray(p_par_base).ravel()
# p_par_sim = np.asarray(p_par_sim).ravel()
bins = np.linspace(p_par_base.min(), p_par_base.max(), 200)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(p_par_base, bins=bins, alpha=0.4, label="All particles 1ppc", log=True)
ax.hist(p_par, bins=bins, label="Low error", log=True, color = "orange")
# ax.hist(p_par_sim, bins=bins, label="Simulated", log=True, color = "blue", alpha=0.5)
ax.set_xlabel(r"$p_{\parallel 0}$")
ax.set_title("Particles with low radial error 1ppc")
ax.legend()
plt.show()



In [ ]:
mask = (err_radial_1ppc > 5*10**(-8))
particles_high_err = np.where(mask)[0]

raw_1ppc = rawdic["gcaNoDrifts1ppc"]["80"]
raw0_1ppc = raw_1ppc[0]
raw_data_1ppc = raw0_1ppc.data

x1_base = raw_data_1ppc["x1"]
x2_base = raw_data_1ppc["x2"]
p1_base = raw_data_1ppc["p1"]
p2_base = raw_data_1ppc["p2"]

x1 = x1_base[particles_high_err]
x2 = x2_base[particles_high_err]
p1 = p1_base[particles_high_err]
p2 = p2_base[particles_high_err]

theo = curvDriftTheo_Raw(raw_1ppc, 1000, direc=1)
p_par = (p1*theo.b1(x1, x2, 0) + p2*theo.b2(x1, x2, 0)) / 1000
p_par_base = (p1_base*theo.b1(x1_base, x2_base, 0) + p2_base*theo.b2(x1_base, x2_base, 0)) / 1000

# p_par_sim = (p1*b1 + p2*b2) / np.sqrt(b1**2 + b2**2)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x1_base, x2_base, alpha=0.5, label="All particles 1ppc")
ax.scatter(x1, x2, alpha=0.5, label="High error")

xmin = min(x1_base.min(), x2_base.min())
xmax = max(x1_base.max(), x2_base.max())
xx = np.linspace(xmin, xmax, 200)
ax.plot(xx, -xx, color="red", linestyle="--", label=r"$x_2=-x_1$")

ax.set_xlabel("x1_0")
ax.set_ylabel("x2_0")
ax.set_title("Particles with high radial error 1ppc")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(p1_base, p2_base, alpha=0.5, label="All particles 1ppc")
ax.scatter(p1, p2, alpha=0.5, label="High error")

ax.set_xlabel("p1_0")
ax.set_ylabel("p2_0")
ax.set_title("Particles with high radial error 1ppc")
ax.legend()
plt.show()



p_par = np.asarray(p_par).ravel()
p_par_base = np.asarray(p_par_base).ravel()
# p_par_sim = np.asarray(p_par_sim).ravel()
bins = np.linspace(p_par_base.min(), p_par_base.max(), 200)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(p_par_base, bins=bins, alpha=0.4, label="All particles 1ppc", log=True)
ax.hist(p_par, bins=bins, label="High error", log=True, color = "orange")
# ax.hist(p_par_sim, bins=bins, label="Simulated", log=True, color = "blue", alpha=0.5)
ax.set_xlabel(r"$p_{\parallel 0}$")
ax.set_title("Particles with high radial error 1ppc")
ax.legend()
plt.show()



In [ ]:
grid = rawdic["gcaNoDrifts"]["80"][0].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in rawdic.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in rawdic[pusher].keys():
        raw_files = rawdic[pusher][dtw]
        if len(raw_files) < 2:
            print(f"Skipping {pusher} nx{dtw}: need at least 2 RAW dumps, found {len(raw_files)}")
            continue

        raw_step = raw_files[1]
        t = raw_step.time[0]
        traj_theo = curvDriftTheo_Raw(raw_files, 1000, direc=1).get_curv_traj(t)

        x1 = raw_step.data["x1"]
        x2 = raw_step.data["x2"]
        x3 = raw_step.data["x3"]

        traj = np.array([x1, x2, x3])

        num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        # num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if not X:
        continue

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error / L")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("ncells")
fig.tight_layout()


In [ ]:
test = "Curv_1step_GcaNoDrifts_dxStudy"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'gcaNoDrifts_DoublePrec': ["24", "40", "60", "80", "100", "120", "140", "160", "180"],
    'gcaNoDrifts_DoublePrec_GcaDoubleprec': ["24", "40", "60", "80", "100", "120", "140", "160", "180"],
    # 'gcaCurvOnly': ["24", "40", "60", "80", "100", "120", "140", "160", "180"],
    'gcaNoDrifts_DoublePrec1ppc': ["24", "40", "60", "80", "100", "120", "140", "160", "180"],
}


sim = createSimDic_nx(path, sim_labels, test)

rawdic = createRawDic_nx(path, sim_labels, test)

In [ ]:
raw_8ppc = rawdic["gcaNoDrifts_DoublePrec"]["80"]
raw_1ppc = rawdic["gcaNoDrifts_DoublePrec1ppc"]["80"]
raw_index_8ppc = 1
raw_index_1ppc = 1

grid = raw_8ppc[0].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]
num_steps = 1


def debug_ranges(label, **arrays):
    print(f"\n{label}")
    for name, values in arrays.items():
        values = np.asarray(values)
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            print(f"  {name}: shape={values.shape}, no finite values")
            continue
        print(
            f"  {name}: shape={values.shape}, "
            f"min={finite.min():.6g}, max={finite.max():.6g}, "
            f"mean={finite.mean():.6g}"
        )


# ---------- 8 ppc ----------
theo_8ppc = curvDriftTheo_Raw(raw_8ppc, 1000, direc=1)
t_8ppc_1 = raw_8ppc[raw_index_8ppc].time[0]


print(f"Time for 8 ppc: {t_8ppc_1}")
traj_theo_8ppc_1 = theo_8ppc.get_curv_traj(t_8ppc_1)

x1_8ppc_1 = raw_8ppc[raw_index_8ppc].data["x1"]
x2_8ppc_1 = raw_8ppc[raw_index_8ppc].data["x2"]
x3_8ppc_1 = raw_8ppc[raw_index_8ppc].data["x3"]
traj_8ppc_1 = np.array([x1_8ppc_1, x2_8ppc_1, x3_8ppc_1])

machine_precision = 10**(-16) / L # double precision machine epsilon in the plot
err_8ppc = np.abs(traj_8ppc_1 - traj_theo_8ppc_1) / L / num_steps
err_radial_8ppc = np.sqrt(err_8ppc[0]**2 + err_8ppc[1]**2)

x1_8ppc_0 = raw_8ppc[0].data["x1"]
x2_8ppc_0 = raw_8ppc[0].data["x2"]
r0_8ppc = np.sqrt(x1_8ppc_0**2 + x2_8ppc_0**2)

p1_8ppc_0 = raw_8ppc[0].data["p1"]
p2_8ppc_0 = raw_8ppc[0].data["p2"]

b1_theo_8ppc_0 = theo_8ppc.b1(x1_8ppc_0, x2_8ppc_0, 0)
b2_theo_8ppc_0 = theo_8ppc.b2(x1_8ppc_0, x2_8ppc_0, 0)

p_par_8ppc = (p1_8ppc_0*b1_theo_8ppc_0 + p2_8ppc_0*b2_theo_8ppc_0) / 1000


# ---------- 1 ppc ----------
theo_1ppc = curvDriftTheo_Raw(raw_1ppc, 1000, direc=1)
t_1ppc_1 = raw_1ppc[raw_index_1ppc].time[0]
traj_theo_1ppc_1 = theo_1ppc.get_curv_traj(t_1ppc_1)

x1_1ppc_1 = raw_1ppc[raw_index_1ppc].data["x1"]
x2_1ppc_1 = raw_1ppc[raw_index_1ppc].data["x2"]
x3_1ppc_1 = raw_1ppc[raw_index_1ppc].data["x3"]
traj_1ppc_1 = np.array([x1_1ppc_1, x2_1ppc_1, x3_1ppc_1])

err_1ppc = np.abs(traj_1ppc_1 - traj_theo_1ppc_1) / L / num_steps
err_radial_1ppc = np.sqrt(err_1ppc[0]**2 + err_1ppc[1]**2)

x1_1ppc_0 = raw_1ppc[0].data["x1"]
x2_1ppc_0 = raw_1ppc[0].data["x2"]
r0_1ppc = np.sqrt(x1_1ppc_0**2 + x2_1ppc_0**2)

p1_1ppc_0 = raw_1ppc[0].data["p1"]
p2_1ppc_0 = raw_1ppc[0].data["p2"]

b1_theo_1ppc_0 = theo_1ppc.b1(x1_1ppc_0, x2_1ppc_0, 0)
b2_theo_1ppc_0 = theo_1ppc.b2(x1_1ppc_0, x2_1ppc_0, 0)

p_par_1ppc = (p1_1ppc_0*b1_theo_1ppc_0 + p2_1ppc_0*b2_theo_1ppc_0) / 1000


print(f"L = {L:.6g}, num_steps = {num_steps}")
print(f"particle counts: 8ppc={len(r0_8ppc)}, 1ppc={len(r0_1ppc)}")
debug_ranges(
    "8ppc initial ranges",
    r0=r0_8ppc,
    p1=p1_8ppc_0,
    p2=p2_8ppc_0,
    b1_theo=b1_theo_8ppc_0,
    b2_theo=b2_theo_8ppc_0,
    p_par=p_par_8ppc,
    err_radial=err_radial_8ppc,
)
debug_ranges(
    "1ppc initial ranges",
    r0=r0_1ppc,
    p1=p1_1ppc_0,
    p2=p2_1ppc_0,
    b1_theo=b1_theo_1ppc_0,
    b2_theo=b2_theo_1ppc_0,
    p_par=p_par_1ppc,
    err_radial=err_radial_1ppc,
)


# error_floor = 1e-15

positive_err = np.concatenate([
    err_radial_8ppc[err_radial_8ppc >= 0],
    err_radial_1ppc[err_radial_1ppc >= 0],
])

if positive_err.size == 0:
    raise ValueError("Log color scale needs at least one positive radial error")

vmin = machine_precision
vmax = positive_err.max()

norm = colors.LogNorm(vmin=vmin, vmax=vmax)
err_radial_8ppc_for_color = np.maximum(err_radial_8ppc, machine_precision)
err_radial_1ppc_for_color = np.maximum(err_radial_1ppc, machine_precision)

fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    r0_1ppc,
    p_par_1ppc,
    label="1ppc",
    alpha=0.5,
    marker="x",
    c=err_radial_1ppc_for_color,
    cmap="viridis",
    norm=norm,
)

ax.set_xlabel("Initial radius")
ax.set_ylabel(r"$p_{\parallel 0}$")
ax.legend()
plt.colorbar(sc, ax=ax, label="Radial error / L")
plt.show()


fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    r0_8ppc,
    p_par_8ppc,
    label="8ppc",
    alpha=0.5,
    c=err_radial_8ppc_for_color,
    cmap="viridis",
    norm=norm,
)

ax.set_xlabel("Initial radius")
ax.set_ylabel(r"$p_{\parallel 0}$")
ax.legend()
plt.colorbar(sc, ax=ax, label="Radial error / L")
plt.show()




fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(r0_8ppc, np.maximum(err_radial_8ppc, machine_precision), label="8ppc", alpha=0.5)
ax.scatter(r0_1ppc, np.maximum(err_radial_1ppc, machine_precision), label="1ppc", alpha=0.5, marker='x')
ax.axhline(machine_precision, color="gray", linestyle=":", linewidth=1, label=r"Machine precision")
ax.set_xlabel("Initial radius")
ax.set_ylabel("Radial error")
ax.legend()
ax.set_yscale("log")
plt.show()

# ---------- log data + same bins ----------
err_radial_8ppc_nonzero = err_radial_8ppc[err_radial_8ppc > 0]
err_radial_1ppc_nonzero = err_radial_1ppc[err_radial_1ppc > 0]

log_err_8ppc = np.log10(err_radial_8ppc_nonzero)
log_err_1ppc = np.log10(err_radial_1ppc_nonzero)

log_min = min(log_err_8ppc.min(), log_err_1ppc.min())
log_max = max(log_err_8ppc.max(), log_err_1ppc.max())

bins = np.linspace(log_min, log_max, 100)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(log_err_8ppc, bins=bins, alpha=0.5, label="8ppc", log=True)
ax.hist(log_err_1ppc, bins=bins, alpha=0.5, label="1ppc", log=True)
ax.axvline(np.log10(machine_precision), color="gray", linestyle=":", linewidth=1, label=r"Machine precision")
ax.set_xlabel(r"$\log_{10}(\mathrm{radial\ error})$")
ax.set_ylabel("Count")
ax.legend()

plt.show()


In [ ]:
mask = (err_radial_1ppc < 10**(-13))
particles_low_err = np.where(mask)[0]

raw_1ppc = rawdic["gcaNoDrifts_DoublePrec1ppc"]["80"]
raw0_1ppc = raw_1ppc[0]
raw_data_1ppc = raw0_1ppc.data

x1_base = raw_data_1ppc["x1"]
x2_base = raw_data_1ppc["x2"]
p1_base = raw_data_1ppc["p1"]
p2_base = raw_data_1ppc["p2"]
p3_base = raw_data_1ppc["p3"]

x1 = x1_base[particles_low_err]
x2 = x2_base[particles_low_err]
p1 = p1_base[particles_low_err]
p2 = p2_base[particles_low_err]
p3 = p3_base[particles_low_err]

theo = curvDriftTheo_Raw(raw_1ppc, 1000, direc=1)
p_par = (p1*theo.b1(x1, x2, 0) + p2*theo.b2(x1, x2, 0)) / 1000
p_par_base = (p1_base*theo.b1(x1_base, x2_base, 0) + p2_base*theo.b2(x1_base, x2_base, 0)) / 1000

p_perp = np.sqrt(((p1 - p1*theo.b1(x1, x2, 0))**2 / 1000 + (p2 - p2*theo.b2(x1, x2, 0))**2 / 1000) + p3**2)
p_perp_base = np.sqrt(((p1_base - p1_base*theo.b1(x1_base, x2_base, 0))**2 / 1000 + (p2_base - p2_base*theo.b2(x1_base, x2_base, 0))**2 / 1000) + p3_base**2)

# p_par_sim = (p1*b1 + p2*b2) / np.sqrt(b1**2 + b2**2)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x1_base, x2_base, alpha=0.5, label="All particles 1ppc")
ax.scatter(x1, x2, alpha=0.5, label="Low error")

xmin = min(x1_base.min(), x2_base.min())
xmax = max(x1_base.max(), x2_base.max())
xx = np.linspace(xmin, xmax, 200)
ax.plot(xx, -xx, color="red", linestyle="--", label=r"$x_2=-x_1$")

ax.set_xlabel("x1_0")
ax.set_ylabel("x2_0")
ax.set_title("Particles with low radial error 1ppc")
ax.legend()
plt.show()



fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(p_par_base, p_perp_base, alpha=0.5, label="All particles 8ppc")
ax.scatter(p_par, p_perp, alpha=0.5, label="High error")

ax.set_xlabel(r"$p_{\parallel 0}$")
ax.set_ylabel(r"$p_{\perp 0}$")
ax.set_title("Particles with low radial error 1ppc")
ax.legend()
plt.show()



fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(p1_base, p2_base, alpha=0.5, label="All particles 1ppc")
ax.scatter(p1, p2, alpha=0.5, label="Low error")

ax.set_xlabel("p1_0")
ax.set_ylabel("p2_0")
ax.set_title("Particles with low radial error 1ppc")
ax.legend()
plt.show()



p_par = np.asarray(p_par).ravel()
p_par_base = np.asarray(p_par_base).ravel()
# p_par_sim = np.asarray(p_par_sim).ravel()
bins = np.linspace(p_par_base.min(), p_par_base.max(), 200)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(p_par_base, bins=bins, alpha=0.4, label="All particles 1ppc", log=True)
ax.hist(p_par, bins=bins, label="Low error", log=True, color = "orange")
# ax.hist(p_par_sim, bins=bins, label="Simulated", log=True, color = "blue", alpha=0.5)
ax.set_xlabel(r"$p_{\parallel 0}$")
ax.set_title("Particles with low radial error 1ppc")
ax.legend()
plt.show()



In [ ]:
mask = (err_radial_8ppc > 10**(-9))
particles_low_err = np.where(mask)[0]

raw_8ppc = rawdic["gcaNoDrifts_DoublePrec"]["80"]
raw0_8ppc = raw_8ppc[0]
raw_data_8ppc = raw0_8ppc.data

x1_base = raw_data_8ppc["x1"]
x2_base = raw_data_8ppc["x2"]
p1_base = raw_data_8ppc["p1"]
p2_base = raw_data_8ppc["p2"]
p3_base = raw_data_8ppc["p3"]

x1 = x1_base[particles_low_err]
x2 = x2_base[particles_low_err]
p1 = p1_base[particles_low_err]
p2 = p2_base[particles_low_err]
p3 = p3_base[particles_low_err]

theo = curvDriftTheo_Raw(raw_8ppc, 1000, direc=1)
p_par = (p1*theo.b1(x1, x2, 0) + p2*theo.b2(x1, x2, 0)) / 1000
p_par_base = (p1_base*theo.b1(x1_base, x2_base, 0) + p2_base*theo.b2(x1_base, x2_base, 0)) / 1000

p_perp = np.sqrt(((p1 - p1*theo.b1(x1, x2, 0))**2 / 1000 + (p2 - p2*theo.b2(x1, x2, 0))**2 / 1000) + p3**2)
p_perp_base = np.sqrt(((p1_base - p1_base*theo.b1(x1_base, x2_base, 0))**2 / 1000 + (p2_base - p2_base*theo.b2(x1_base, x2_base, 0))**2 / 1000) + p3_base**2)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x1_base, x2_base, alpha=0.5, label="All particles 8ppc")
ax.scatter(x1, x2, alpha=0.5, label="High error")

xmin = min(x1_base.min(), x2_base.min())
xmax = max(x1_base.max(), x2_base.max())
xx = np.linspace(xmin, xmax, 200)
ax.plot(xx, -xx, color="red", linestyle="--", label=r"$x_2=-x_1$")

ax.set_xlabel("x1_0")
ax.set_ylabel("x2_0")
ax.set_title("Particles with high radial error 8ppc")
ax.legend()
plt.show()



fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(p_par_base, p_perp_base, alpha=0.5, label="All particles 8ppc")
ax.scatter(p_par, p_perp, alpha=0.5, label="High error")

ax.set_xlabel(r"$p_{\parallel 0}$")
ax.set_ylabel(r"$p_{\perp 0}$")
ax.set_title("Particles with high radial error 8ppc")
ax.legend()
plt.show()



p_par = np.asarray(p_par).ravel()
p_par_base = np.asarray(p_par_base).ravel()
# p_par_sim = np.asarray(p_par_sim).ravel()
bins = np.linspace(p_par_base.min(), p_par_base.max(), 200)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(p_par_base, bins=bins, alpha=0.4, label="All particles 8ppc", log=True)
ax.hist(p_par, bins=bins, label="High error", log=True, color = "orange")
# ax.hist(p_par_sim, bins=bins, label="Simulated", log=True, color = "blue", alpha=0.5)
ax.set_xlabel(r"$p_{\parallel 0}$")
ax.set_title("Particles with high radial error 8ppc")
ax.legend()
plt.show()



In [ ]:
grid = rawdic["gcaNoDrifts_DoublePrec"]["80"][0].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in rawdic.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in rawdic[pusher].keys():
        raw_files = rawdic[pusher][dtw]
        if len(raw_files) < 2:
            print(f"Skipping {pusher} nx{dtw}: need at least 2 RAW dumps, found {len(raw_files)}")
            continue

        raw_step = raw_files[1]
        t = raw_step.time[0]
        traj_theo = curvDriftTheo_Raw(raw_files, 1000, direc=1).get_curv_traj(t)

        x1 = raw_step.data["x1"]
        x2 = raw_step.data["x2"]
        x3 = raw_step.data["x3"]

        traj = np.array([x1, x2, x3])

        num_steps = 1
        # num_steps = float(dtw.replace("_", ".")) * 100 # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 0.01)
        # num_steps = float(dtw.replace("_", ".")) / 1000  # divide so it is evaluated error per step accounting for the diffferences in time (bring everythin the smae as dtw 1000)
        err = (np.abs(traj - traj_theo)) / L / num_steps 

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if not X:
        continue

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error / L")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("ncells")
fig.tight_layout()


In [ ]:
# test = "Curv"
# path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
# sim_labels = {
#     'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
#     'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
# }

# B = 1000.
# sim = createSimDic(path, sim_labels, test)

# test = "Curv_dx"
# path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
# sim_labels = {
#     'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
#     'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
# }

# B = 1000.
# sim2 = createSimDic(path, sim_labels, test)

# sims = {"80": sim, "120": sim2}

In [ ]:


# components = ["r", "x3"]
# fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)

# pushers = []
# for dx in sims.keys():
#     for pusher in sims[dx].keys():
#         if pusher not in pushers:
#             pushers.append(pusher)

# color_map = {pusher: f"C{i}" for i, pusher in enumerate(pushers)}
# style_map = {"120": "--"}
# for dx in sims.keys():
#     linestyle = style_map.get(dx, '-')
#     t = sims[dx]["Gca"]["1000"]["test_electrons"]["tracks"]["t"][0,1]
#     traj_theo = curvDriftTheo(sims[dx]["Gca"]["1000"], B).get_curv_traj(t)[:,:,0]

#     grid = sims[dx]["Gca"]["1000"]["test_electrons"]["tracks"].grid
#     grid = [float(grid[0, 0]), float(grid[0, 1])]
#     L = grid[1] - grid[0]

#     for pusher in sims[dx].keys():
#         X = []    
#         Y = []
#         YERR = []    
#         YMax = []
#         for dtw in sims[dx][pusher].keys():
#             x1 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
#             x2 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
#             x3 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

#             traj = np.array([x1, x2, x3])

#             err = (np.abs(traj - traj_theo)) / L

#             err_radial = np.sqrt(err[0]**2 + err[1]**2)

#             err = np.array([err_radial, err[2]])

#             mean = np.mean(err, axis=1)
#             std = np.std(err, axis=1, ddof=1)

#             X.append(float(dtw.replace("_", ".")))
#             Y.append(mean)
#             YERR.append(std)
#             YMax.append(np.max(err, axis=1))
        
#         X = np.asarray(X, dtype=float)
#         Y = np.asarray(Y, dtype=float)
#         YERR = np.asarray(YERR, dtype=float)
#         YMax = np.asarray(YMax, dtype=float)

#         for i, ax in enumerate(axes):
#             eb = ax.errorbar(
#                 X, Y[:, i], yerr=YERR[:, i],
#                 fmt='o',
#                 markersize=5,
#                 linestyle=linestyle,
#                 linewidth=1.2,
#                 capsize=3,
#                 elinewidth=1.0,
#                 color=color_map[pusher],
#             )

#             ax.plot(
#                 X,
#                 YMax[:, i],
#                 marker='x',
#                 linestyle='none',
#                 markersize=5,
#                 markeredgewidth=1.0,
#                 zorder=3,
#                 color=eb.lines[0].get_color(),
#             )
#             ax.set_ylabel(fr"{components[i]} error / L")
#             ax.set_xscale("log")
#             ax.set_yscale("log")
#             ax.grid(True, alpha=0.3)

# pusher_handles = [
#     Line2D([0], [0], color=color_map[pusher], marker='o', linestyle='-', label=pusher)
#     for pusher in pushers
# ]
# style_handles = [
#     Line2D([0], [0], color='black', linestyle=style_map.get(dx, '-'), label=fr"{dx} cells")
#     for dx in sims.keys()
# ]

# axes[0].set_title("t = " + str(round(t * B / 2.0 / np.pi, 1)) + r" $[2\pi / \Omega_e]$")
# axes[0].legend(handles=pusher_handles + style_handles)
# axes[-1].set_xlabel("dtw")
# fig.tight_layout()


In [ ]:
# test = "Curv_1step"
# path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
# sim_labels = {
#     'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
#     'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
# }

# B = 1000.
# sim = createSimDic(path, sim_labels, test)

# test = "Curv_dx_1step"
# path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
# sim_labels = {
#     'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
#     'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
#     # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
# }

# B = 1000.
# sim2 = createSimDic(path, sim_labels, test)

# sims = {"80": sim, "120": sim2}

In [ ]:
# components = ["r", "x3"]
# fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)

# pushers = []
# for dx in sims.keys():
#     for pusher in sims[dx].keys():
#         if pusher not in pushers:
#             pushers.append(pusher)

# color_map = {pusher: f"C{i}" for i, pusher in enumerate(pushers)}
# style_map = {"120": "--"}
# for dx in sims.keys():
#     linestyle = style_map.get(dx, '-')

#     for pusher in sims[dx].keys():
#         X = []    
#         Y = []
#         YERR = []    
#         YMax = []
#         for dtw in sims[dx][pusher].keys():
#             t = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
#             traj_theo = curvDriftTheo(sims[dx][pusher][dtw], B).get_curv_traj(t)[:,:,0]

#             grid = sims[dx][pusher][dtw]["test_electrons"]["tracks"].grid
#             grid = [float(grid[0, 0]), float(grid[0, 1])]
#             L = grid[1] - grid[0]

#             x1 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
#             x2 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
#             x3 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

#             traj = np.array([x1, x2, x3])

#             err = (np.abs(traj - traj_theo)) / L

#             err_radial = np.sqrt(err[0]**2 + err[1]**2)

#             err = np.array([err_radial, err[2]])

#             mean = np.mean(err, axis=1)
#             std = np.std(err, axis=1, ddof=1)

#             X.append(float(dtw.replace("_", ".")))
#             Y.append(mean)
#             YERR.append(std)
#             YMax.append(np.max(err, axis=1))
        
#         X = np.asarray(X, dtype=float)
#         Y = np.asarray(Y, dtype=float)
#         YERR = np.asarray(YERR, dtype=float)
#         YMax = np.asarray(YMax, dtype=float)

#         for i, ax in enumerate(axes):
#             eb = ax.errorbar(
#                 X, Y[:, i], yerr=YERR[:, i],
#                 fmt='o',
#                 markersize=5,
#                 linestyle=linestyle,
#                 linewidth=1.2,
#                 capsize=3,
#                 elinewidth=1.0,
#                 color=color_map[pusher],
#             )

#             ax.plot(
#                 X,
#                 YMax[:, i],
#                 marker='x',
#                 linestyle='none',
#                 markersize=5,
#                 markeredgewidth=1.0,
#                 zorder=3,
#                 color=eb.lines[0].get_color(),
#             )
#             ax.set_ylabel(fr"{components[i]} error / L")
#             ax.set_xscale("log")
#             ax.set_yscale("log")
#             ax.grid(True, alpha=0.3)

# pusher_handles = [
#     Line2D([0], [0], color=color_map[pusher], marker='o', linestyle='-', label=pusher)
#     for pusher in pushers
# ]
# style_handles = [
#     Line2D([0], [0], color='black', linestyle=style_map.get(dx, '-'), label=fr"{dx} cells")
#     for dx in sims.keys()
# ]

# axes[0].set_title("1st step")
# axes[0].legend(handles=pusher_handles + style_handles)
# axes[-1].set_xlabel("dtw")
# fig.tight_layout()


### Gca - no GradB

In [ ]:
sim = ou.Simulation("/home/exxxx5/Tese/Decks/StudyConvergence/Curv_GcaNoGradB/Gca/dtw1/Gca.in")
sim["test_electrons"]["tracks"].load_all()

particle = 0

grid = sim["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

theo = curvDriftTheo(sim, 1000, direc = 1)

traj_theo = theo.get_curv_traj(sim["test_electrons"]["tracks"]["t"][0,:])[:,particle,:]

In [ ]:
t = sim["test_electrons"]["tracks"]["t"][particle, :]

traj_sim = np.array([
    sim["test_electrons"]["tracks"]["x1"][particle, :],
    sim["test_electrons"]["tracks"]["x2"][particle, :],
    sim["test_electrons"]["tracks"]["x3"][particle, :],
])

err = np.abs(traj_sim - traj_theo) / L
err_tot = np.sqrt(np.sum((traj_sim - traj_theo)**2, axis=0)) / L

components = ["x1", "x2", "x3"]
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

for i, comp in enumerate(components):
    axes[i].plot(t, err[i], label=fr"$|{comp}_{{sim}} - {comp}_{{theo}}| / L$")
    axes[i].set_ylabel(fr"{comp} err / L")
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

axes[3].plot(t, err_tot, color="black", label=r"$||\mathbf{x}_{sim} - \mathbf{x}_{theo}|| / L$")
axes[3].set_ylabel(r"total err / L")
axes[3].set_xlabel("t")
axes[3].grid(True, alpha=0.3)
axes[3].legend()

fig.suptitle(f"Particle {particle} trajectory error over time")
fig.tight_layout()


In [ ]:
t = sim["test_electrons"]["tracks"]["t"][particle, 1:]
p1 = sim["test_electrons"]["tracks"]["p1"][particle, 1:]
p2 = sim["test_electrons"]["tracks"]["p2"][particle, 1:]
p3 = sim["test_electrons"]["tracks"]["p3"][particle, 1:]
ptotal = np.sqrt(p1**2 + p2**2 + p3**2)

momenta = [p1, p2, p3, ptotal]
labels = ["p1", "p2", "p3", "|p|"]

fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

for ax, p, label in zip(axes, momenta, labels):
    ax.plot(t, p, label=label)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel("t")
fig.suptitle(f"Particle {particle} momentum over time")
fig.tight_layout()


In [ ]:
# p3 error
t = sim["test_electrons"]["tracks"]["t"][particle, 1:]
p3_sim = sim["test_electrons"]["tracks"]["p3"][particle, 1:]



vc, v_par = theo._curv_v()

p3_theo = vc[particle] * theo.gamma_0[particle]

p3_err = np.abs(p3_sim - p3_theo)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(t, p3_sim, label=r"$p_{3,sim}$")
axes[0].axhline(p3_theo, color="black", linestyle="--", label=fr"$p_{{3,theo}} = {p3_theo:.6g}$")
axes[0].set_ylabel(r"$p_3$")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(t, p3_err, color="tab:red", label=r"$|p_{3,sim} - p_{3,theo}|$")
axes[1].set_xlabel("t")
axes[1].set_ylabel(r"$p_3$ error")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

fig.suptitle(f"Particle {particle} p3 comparison")
fig.tight_layout()
